**🟢 Basic Level – Fashion-MNIST**



```
1. Carga del Dataset
```



In [ ]:
from tensorflow.keras.datasets import fashion_mnist
import tensorflow as tf
import numpy as np

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Usar solo una fracción (1000 muestras)
x_train = x_train[:1000]
y_train = y_train[:1000]
x_test = x_test[:200]
y_test = y_test[:200]


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step




```
2. Preprocesamiento
```



In [ ]:
# Redimensionar a 96x96 porque MobileNetV2 requiere imágenes mayores a 32x32
x_train = tf.image.resize(tf.expand_dims(x_train, -1), [96, 96]) / 255.0
x_test = tf.image.resize(tf.expand_dims(x_test, -1), [96, 96]) / 255.0

# Repetir canales para tener 3 (como imágenes RGB)
x_train = tf.repeat(x_train, 3, axis=-1)
x_test = tf.repeat(x_test, 3, axis=-1)


```
3. Definición del Modelo con MobileNetV2
```



In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input

def build_model():
    base = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights=None)
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(10, activation='softmax')(x)
    model = Model(inputs=base.input, outputs=x)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model



```
4. Entrenamiento en CPU
```



In [ ]:
import time

with tf.device('/CPU:0'):
    model_cpu = build_model()
    start_cpu = time.time()
    model_cpu.fit(x_train, y_train, epochs=5, batch_size=32, verbose=2)
    end_cpu = time.time()
    cpu_time = end_cpu - start_cpu


Epoch 1/5
32/32 - 290s - 9s/step - accuracy: 0.3600 - loss: 1.7753
Epoch 2/5
32/32 - 230s - 7s/step - accuracy: 0.6360 - loss: 1.0192
Epoch 3/5
32/32 - 266s - 8s/step - accuracy: 0.7210 - loss: 0.7749
Epoch 4/5
32/32 - 259s - 8s/step - accuracy: 0.7760 - loss: 0.6174
Epoch 5/5
32/32 - 220s - 7s/step - accuracy: 0.8010 - loss: 0.5547




```
5. Entrenamiento en GPU
```



In [ ]:
with tf.device('/GPU:0'):
    model_gpu = build_model()
    start_gpu = time.time()
    model_gpu.fit(x_train, y_train, epochs=5, batch_size=32, verbose=2)
    end_gpu = time.time()
    gpu_time = end_gpu - start_gpu


Epoch 1/5
32/32 - 68s - 2s/step - accuracy: 0.3690 - loss: 1.7087
Epoch 2/5
32/32 - 1s - 31ms/step - accuracy: 0.6050 - loss: 1.0874
Epoch 3/5
32/32 - 1s - 27ms/step - accuracy: 0.6730 - loss: 0.8609
Epoch 4/5
32/32 - 1s - 39ms/step - accuracy: 0.7680 - loss: 0.6521
Epoch 5/5
32/32 - 1s - 39ms/step - accuracy: 0.8340 - loss: 0.4714




```
Resultados Finales
```



In [ ]:
print(f"⏱️ Tiempo CPU: {cpu_time:.2f} segundos")
print(f"⚡ Tiempo GPU: {gpu_time:.2f} segundos")
print(f"🚀 Aceleración: {cpu_time / gpu_time:.2f}x")


⏱️ Tiempo CPU: 1306.97 segundos
⚡ Tiempo GPU: 72.12 segundos
🚀 Aceleración: 18.12x


**Conclusiones:**


* El entrenamiento con GPU fue 18 veces más rápido que con CPU.

* Este resultado valida que el uso de GPU es altamente recomendable para acelerar procesos de entrenamiento en Deep Learning.

* Aunque el objetivo no era mejorar la precisión, el uso de MobileNetV2 en un subconjunto del dataset mostró que se puede comparar el rendimiento sin perder generalidad.

* Este tipo de análisis es crucial para tomar decisiones sobre recursos computacionales en proyectos reales.


